# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'jhaff2'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
d = load('p1a_extinction')
X  = d['airmass'].to_numpy()
m  = d['zeropoint_mag'].to_numpy()
sig = d['sigma_quoted_mag'].to_numpy()
N, M = len(X), 2

# 1. Weighted least squares by hand: beta = (A^T W A)^-1 A^T W y
A = np.vstack([X, np.ones(N)]).T
W = np.diag(1/sig**2)
C = np.linalg.inv(A.T @ W @ A)          # covariance from quoted errors alone
k, m0 = C @ A.T @ W @ m
sig_k_quoted = np.sqrt(C[0, 0])

# 2. polyfit
p, Cp = np.polyfit(X, m, 1, w=1/sig, cov=True)
sig_k_polyfit = np.sqrt(Cp[0, 0])
p2, Cp2 = np.polyfit(X, m, 1, w=1/sig, cov='unscaled')   # for comparison

# 3. chi^2
resid = (m - (k*X + m0)) / sig
chi2 = np.sum(resid**2)
dof = N - M
p_large = stats.chi2.sf(chi2, dof)      # P(chi2' >= chi2)
p_small = stats.chi2.cdf(chi2, dof)     # P(chi2' <= chi2)

print(f"k = {k:.4f} +/- {sig_k_quoted:.4f} mag/airmass (quoted errors)")
print(f"polyfit sigma_k = {sig_k_polyfit:.4f}, ratio = {sig_k_polyfit/sig_k_quoted:.3f}")
print(f"sqrt(chi2/dof) = {np.sqrt(chi2/dof):.3f}")
print(f"chi2 = {chi2:.2f}, dof = {dof}, P(>=) = {p_large:.3g}, P(<=) = {p_small:.3g}")

**ANSWER:**

2. cov=True multiplies the covariance by the reduced χ², which is chi2/13, or 5.21/13 = 0.40 in numbers. σ_k drops by √0.40 = 0.633 ratio, and the error also scales from the square root of that number, from 0.0093 to 0.0059. Treats weights of error bars as relative, not absolute. Error is set by scatter of data, not pipeline error bars. 
Claude checked this by looking at the ratio and the sqrt of chi2/dof, and they were the same so the check is good

3. x**2 is 5.21 for `13 degrees of freedom. The prob of getting a bound this low is  0.0296 or 3%, while a 97% chance that we could get bounds being high enough. So, its large enough, but only a 3% chance it could be this low. SMall x tail matters, shows that errors could be over estimated. X^2 and residuals are small if sigma is too big, and it being too small of x^2 means it is likely overestimating.

4. Overestimated pipeline errors to lead to this overestimation. I would go with 0.0059 for sigma, as it is 100 - 63.3% smaller tham the higher one, and we know the error bars are overestimating. Trusting the scatter of the data over the the raw sigma point values is likely more accurate.

Plotted a residual chart, points show no trend

In [ ]:
resid_mag = m - (k*X + m0)          # residuals in mag
resid_norm = resid_mag / sig        # in units of quoted sigma

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

# Left: residuals in mag, with the quoted error bars
ax1.errorbar(X, resid_mag, yerr=sig, fmt='o', capsize=3)
ax1.axhline(0, color='k', lw=1)
ax1.set_xlabel('Airmass X')
ax1.set_ylabel('Residual (mag)')
ax1.set_title('Residuals with quoted errors')

# Right: normalized residuals, with +/-1 and +/-2 sigma guides
ax2.plot(X, resid_norm, 'o')
ax2.axhline(0, color='k', lw=1)
for s in (1, -1):
    ax2.axhline(s, color='gray', ls='--', lw=1)
for s in (2, -2):
    ax2.axhline(s, color='gray', ls=':', lw=1)
ax2.set_xlabel('Airmass X')
ax2.set_ylabel('Residual / $\\sigma_{quoted}$')
ax2.set_title('Normalized residuals')

plt.tight_layout()
plt.show()

print(f"RMS of normalized residuals = {np.sqrt(np.mean(resid_norm**2)):.2f}")
print(f"Points outside +/-1 sigma: {np.sum(np.abs(resid_norm) > 1)} of {N}")

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
d = load('p1b_centroid')
print(d.columns.tolist(), len(d))   # check the layout first

# Adjust names if your columns differ. These come from the problem statement.
N     = d['n_photons'].to_numpy()[0]
sigma = d['psf_sigma_arcsec'].to_numpy()[0]
quoted = d['quoted_sigma_x0_arcsec'].to_numpy()[0]

crb = sigma / np.sqrt(N)
print(f"N = {N}, PSF sigma = {sigma:.3f} arcsec")
print(f"Cramer-Rao bound = {crb:.4f} arcsec")
print(f"Quoted error      = {quoted:.4f} arcsec")
print(f"quoted / CRB      = {quoted/crb:.3f}")

**ANSWER:**

2. 0.031 is blow the bound of 0.0455, with a ratio of 0.68. There might have been assumed to be more points than actually had, (higher N would lower the error, makes it more certian) or estimater is baised to hide a larger error. The PSF width could have also been wrong, and they could have cut or not weighted outlier points to make that possible. That would also make the error seem smaller, as farther out points wouldn't matter as much to a system that didn't weight enough points. 

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
d = load('p1c_profile')
dep = d['depth_ppt'].to_numpy()
m2l = d['minus2lnL'].to_numpy()

# Best depth: fit a parabola to points near the minimum (more robust than finite differences)
i0 = np.argmin(m2l)
near = m2l - m2l[i0] < 4            # points within Delta(-2lnL) < 4 of the minimum
c2, c1, c0 = np.polyfit(dep[near], m2l[near], 2)
best = -c1 / (2*c2)
fpp = 2*c2                           # second derivative of -2lnL
sigma = np.sqrt(2/fpp)
sigma_classmate = 1/np.sqrt(fpp)

print(f"grid minimum at {dep[i0]:.4f} ppt; parabola minimum at {best:.4f} ppt")
print(f"sigma (correct)   = {sigma:.4f} ppt")
print(f"sigma (classmate) = {sigma_classmate:.4f} ppt, ratio = {sigma_classmate/sigma:.3f}")

In [ ]:
y = m2l - m2l.min()
ib = np.argmin(y)

# Left crossing (y decreasing toward the minimum, so reverse for np.interp)
left  = np.interp(1, y[:ib+1][::-1], dep[:ib+1][::-1])
right = np.interp(1, y[ib:], dep[ib:])
dbest = dep[ib]
print(f"Delta=1 interval: [{left:.4f}, {right:.4f}] ppt")
print(f"best = {dbest:.4f}  -{dbest-left:.4f} / +{right-dbest:.4f} ppt")

In [ ]:
rng = np.random.default_rng(457)
truth = best                               # any value works; coverage doesn't depend on it
fits = rng.normal(truth, sigma, size=5000)  # fake best-fit depths using the CORRECT sigma

for label, s in [('correct sigma', sigma), ('classmate sigma', sigma_classmate)]:
    cover = np.mean(np.abs(fits - truth) < s)
    err = np.sqrt(cover*(1-cover)/len(fits))
    print(f"{label}: coverage = {cover:.3f} +/- {err:.3f}")

print("predicted:", stats.norm.cdf(1) - stats.norm.cdf(-1), stats.norm.cdf(1/np.sqrt(2)) - stats.norm.cdf(-1/np.sqrt(2)))

**ANSWER:**

1. sigma (correct)   = 0.3363 ppt
sigma (classmate) = 0.2378 ppt, ratio = 0.707
Classmate is wrong, it is underestimated. They forgot a sqrt of 2 which should replace the 1 ontop of the fraction.off by this factor. 0.707 is 1/sqrt2, further proving this (0.707 is the ratio) 
2. The two half widths seem to be similar, so they can be trsuted.
3. I predicted it to be right about 70% of the time based on vibes, and I was wrong. It was only accurate 52% of the time, while the correct sigma was accurate 66% of the time. Claude predictions were much mroe accurate. 

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
d = load('p1d_mass_samples')
M = d['M2_Msun'].to_numpy()
n = len(M)

lo, med, hi = np.percentile(M, [16, 50, 84])
print(f"M2 = {med:.4f} +{hi-med:.4f} -{med-lo:.4f} Msun")

std = M.std(ddof=1)
print(f"variance = {M.var(ddof=1):.5f} Msun^2")
print(f"84th - 16th = {hi-lo:.4f} Msun   (half = {(hi-lo)/2:.4f})")
print(f"std = {std:.4f} Msun,  std/sqrt(n) = {std/np.sqrt(n):.5f} Msun")
print(f"(84-16)/std = {(hi-lo)/std:.3f},  1/sqrt(n) = {1/np.sqrt(n):.4f}")
print(f"skew = {stats.skew(M):.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(M, bins=80, density=True, alpha=0.6)
for v, ls in [(lo, '--'), (med, '-'), (hi, '--')]:
    ax.axvline(v, color='k', ls=ls)
# What a symmetric +/- std would claim, for comparison
ax.axvspan(M.mean()-std, M.mean()+std, color='C1', alpha=0.2, label='mean ± std')
ax.set_xlabel('$M_2$ ($M_\\odot$)'); ax.set_ylabel('Posterior density')
ax.legend(); plt.show()

**ANSWER:**

2. i) is sigma^2, with units of M solar^2. It shouldn'tbe squared. It isn't a range so its not an error bar. Cant trust. sigma is off by 1.47 only if you ignore that. 
ii) It is the full width of the 68% interval, or 2.77 solar masses. it is 1.88 ratio when divided by the std relative to sigma. 
iii) 0.016 solar masses is too small, this is monte carlo stuff to divide by sqrt of n, not sigma. off by 1/sqrt 8000

3. No, as there is a tail that leans toward the higher range, though has a smaller density. Towards the lower range has a higehr density, but it covers less iof the spread. 

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
d = load('p1e_slope_intercept_samples')
print(d.columns.tolist())             # check the names before using them
s = d.iloc[:, 0].to_numpy()           # slope column (adjust if the order differs)
b = d.iloc[:, 1].to_numpy()           # intercept column

sig_marg = s.std(ddof=1)
rho = np.corrcoef(s, b)[0, 1]

# Fixed intercept: spread of the slope in a thin slice around the best intercept
b_best = np.median(b)
w = 0.1 * b.std()
sl = np.abs(b - b_best) < w
sig_fixed_slice = s[sl].std(ddof=1)

# Analytic, from the sample covariance: 1/sqrt(precision matrix element)
C = np.cov(s, b)
sig_fixed = 1/np.sqrt(np.linalg.inv(C)[0, 0])

print(f"marginal  sigma_s = {sig_marg:.4f}")
print(f"fixed (analytic)  = {sig_fixed:.4f},  fixed (slice, n={sl.sum()}) = {sig_fixed_slice:.4f}")
print(f"ratio fixed/marg = {sig_fixed/sig_marg:.3f},  sqrt(1-rho^2) = {np.sqrt(1-rho**2):.3f}")
print(f"rho = {rho:.3f}")
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(s, b, '.', ms=2, alpha=0.3)
s0, b0 = np.median(s), np.median(b)
ax.errorbar(s0, b0 - 3*b.std(), xerr=sig_marg, fmt='o', color='C3', capsize=4, label='marginal')
ax.errorbar(s0, b0 - 3.5*b.std(), xerr=sig_fixed, fmt='s', color='C2', capsize=4, label='fixed intercept')
ax.axhline(b_best, color='gray', lw=0.8, ls=':')
ax.set_xlabel('slope'); ax.set_ylabel('intercept'); ax.legend(); plt.show()

**ANSWER:**
2. error too small by ratio of 0.335, if rho = 0, they would agree and it would be ratio of 1. the slope clearly changes based on the intercepotm so treating it as fixed is wrong unless we for sure know what it is. Rho is -0.942, not 0. ratio is ratio = sqrt(1-rho^2). ratio is also fixed sigma/marginal sigma. These two both equal eachother. 

3. It would only be right if we knew the intercept for sure, so that we could ignore the uncertainty in order to fix the intercept. 

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
from astropy.cosmology import FlatLambdaCDM
cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

d = load('p2_pantheon_sne')
print(d.shape, d.columns.tolist())

z, mB, x1, c = (d[k].to_numpy() for k in ['z', 'mB', 'x1', 'c'])
logM = d['host_logmass'].to_numpy()
mu   = cosmo.distmod(z).value                       # mag
step = (logM > 10).astype(float)                    # Theta
hub  = mB - mu                                      # Hubble residual (mag)

c_kms = 299792.458
sig_pec = 5/np.log(10) * 250/(c_kms*z)              # mag

N = len(z)
Cov = np.zeros((N, 3, 3))
Cov[:,0,0], Cov[:,1,1], Cov[:,2,2] = d['mB_err']**2, d['x1_err']**2, d['c_err']**2
Cov[:,0,1] = Cov[:,1,0] = d['cov_mB_x1']
Cov[:,0,2] = Cov[:,2,0] = d['cov_mB_c']
Cov[:,1,2] = Cov[:,2,1] = d['cov_x1_c']
print('all C_i positive definite:', np.all(np.linalg.eigvalsh(Cov) > 0))

dat = dict(z=z, mB=mB, x1=x1, c=c, mu=mu, step=step, Cov=Cov, sig_pec=sig_pec)
def subset(dat, keep): return {k: v[keep] for k, v in dat.items()}

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(17, 3.8))
for a_, (v, lab) in zip(ax[:3], [(x1, '$x_1$'), (c, '$c$'), (logM, 'host $\\log_{10}(M_\\star/M_\\odot)$')]):
    a_.plot(v, hub, '.', alpha=.6)
    s = stats.linregress(v, hub)
    a_.set_title(f'slope = {s.slope:.3f}'); a_.set_xlabel(lab); a_.set_ylabel('$m_B-\\mu$ (mag)')
ax[3].plot(logM, x1, '.', alpha=.6); ax[3].set_xlabel('host log mass'); ax[3].set_ylabel('$x_1$')
plt.tight_layout(); plt.show()
print('mean x1, low-mass / high-mass hosts:', x1[step==0].mean(), x1[step==1].mean())

**ANSWER:**

1. mean x1, low-mass / high-mass hosts: 0.2932933113207547 -0.4213810972222223
2. The slope for c is positive, as redder means it is fainter, and higher c means more redshift. It becomes more pos (mag wise) and more pos mag is dimmer. while the slope of x1 is -0.166, negative. High strech, incr x1, is brighter, so the mB will be smaller as mag is smaller when brighter. Tripp expects. 
3. x1 is correlated to host mass, higher host mass is brighter, but shorter lived supernova. Negative correlation, host mass is negatively correlated to x1. Host mass also having loose correlations to brightness and c also causes
4. there are some points that look like outlyers. Two are high in the c plot, there could be some in the mass plot, and maybe some in the x1.  
5. x1 and c dimmesnionless, so a and b are mag per x1 per c

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
def resid_var(dat, M, a, b, g, lns):
    r = dat['mB'] - (M + dat['mu'] - a*dat['x1'] + b*dat['c'] + g*dat['step'])
    C = dat['Cov']
    v = (C[:,0,0] + a*a*C[:,1,1] + b*b*C[:,2,2] + 2*a*C[:,0,1]
         - 2*b*C[:,0,2] - 2*a*b*C[:,1,2] + dat['sig_pec']**2 + np.exp(2*lns))
    return r, v

def make_model(dat, with_gamma=True, gmax=0.3):
    lo = np.array([-21, -1, -1] + ([-gmax] if with_gamma else []) + [np.log(1e-3)])
    hi = np.array([-17,  1,  8] + ([ gmax] if with_gamma else []) + [0.0])
    logvol = np.sum(np.log(hi - lo))
    def lnlike(p):
        g = p[3] if with_gamma else 0.0
        r, v = resid_var(dat, p[0], p[1], p[2], g, p[-1])
        return -0.5*np.sum(r*r/v + np.log(2*np.pi*v))
    def lnpost(p):
        p = np.asarray(p)
        if np.any(p < lo) or np.any(p > hi): return -np.inf
        return lnlike(p) - logvol
    return lnlike, lnpost, lo, hi

lnlike, lnpost, lo, hi = make_model(dat)
x0 = np.array([-19.3, 0.14, 3.0, -0.05, np.log(0.1)])
res = minimize(lambda p: -lnlike(p), x0, method='L-BFGS-B', bounds=list(zip(lo, hi)))
M_, a_, b_, g_, lns_ = res.x
r, v = resid_var(dat, *res.x)
pull = r/np.sqrt(v)
for i in np.argsort(-np.abs(pull))[:10]:
    print(i, f'pull = {pull[i]:6.1f}   r = {r[i]:6.2f} mag')

**ANSWER:**
mb, x1, and c have sources of error that come from the same curve, so they are correlated, and all the other erros stem from that. 


1. $$\mathbf v^TC\mathbf v=\sigma_{m_B}^2+\alpha^2\sigma_{x_1}^2+\beta^2\sigma_c^2+2\alpha\,{\rm cov}(m_B,x_1)-2\beta\,{\rm cov}(m_B,c)-2\alpha\beta\,{\rm cov}(x_1,c)$$
   From Claude
2. $$\ln L=-\frac12\sum_i\left[\frac{r_i^2}{\sigma_i^2}+\ln\!\left(2\pi\sigma_i^2\right)\right]$$
Also from claude

3. M has a prior -20, -18 mag, as this captures the range. a has a prior -1 to 1, as it fits within the publichsed range and should allow for enough wiggle room. betta would be from -1 to 8, as the published value is around 3, so that is similar to why the ranges for a were chosen. For gamma, I choose -0.3 to 0.3, as the null would be gamma=0, and 5x 0.06 (published) is 0.3, and that would be clear in the residuals. ln σ_int would be 0.0001 to 1, as, since it is a log value, it could be anywhere on this scale. 

4. ln σ_int over σ_int. σ_int is a positive scale parameter that could span orders of magnitude. Sampling its logarithm enforces positivity with no hard edge at zero, and a flat prior in ln σ_int gives equal weight per decade. The posterior median is ln σ_int ≈ −1.88 (σ_int ≈ 0.15 mag), far from the lower bound at ln 0.001 = −6.9, so the result doesn’t depend on that bound.


5. pull 56 and 107 is clearly at least 2 of the outliers. The rest seems to blend with noise, but 21 pull and 242 pull are suspecious. I'm going with only the first two.I will be removing them. they were mutliple sigma above 3, (6.7, 6.1), and just by looking at it I found them.This would even clear a 6 sigma cutoff, so it definietly is clean and not removing good data. 

In [ ]:
bad = np.array([56, 107])
keep = np.ones(N, bool); keep[bad] = False
D = subset(dat, keep)

# refit on the cleaned set
lnlike, lnpost, lo, hi = make_model(D)
res2 = minimize(lambda p: -lnlike(p), res.x, method='L-BFGS-B', bounds=list(zip(lo, hi)))
r2, v2 = resid_var(D, *res2.x); pull2 = r2/np.sqrt(v2)
print('before:', np.round(res.x[:4], 3), 'sigma_int =', np.exp(res.x[4]))
print('after: ', np.round(res2.x[:4], 3), 'sigma_int =', np.exp(res2.x[4]))
print('largest |pull| now:', np.round(np.sort(np.abs(pull2))[::-1][:5], 1))

# the removed objects, scored against the cleaned model
rb, vb = resid_var(subset(dat, ~keep), *res2.x)
print('removed objects, pull under cleaned fit:', np.round(rb/np.sqrt(vb), 1))

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
keep = np.ones(N, bool)            # <-- set False for the contaminants you removed
D = subset(dat, keep)
lnlike, lnpost, lo, hi = make_model(D)

# MAP from several starts
rng = np.random.default_rng(457)
fits = [minimize(lambda p: -lnpost(p) if np.isfinite(lnpost(p)) else 1e10,
                 x0 + rng.normal(0, [0.2, .1, .5, .05, .5]), method='Nelder-Mead',
                 options=dict(maxiter=20000, xatol=1e-6, fatol=1e-8)) for _ in range(5)]
best = min(fits, key=lambda f: f.fun); theta_map = best.x
print([round(f.fun, 4) for f in fits])      # should agree

def hessian(f, x, h):
    n = len(x); H = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            ei = np.zeros(n); ej = np.zeros(n); ei[i] = h[i]; ej[j] = h[j]
            H[i,j] = H[j,i] = (f(x+ei+ej) - f(x+ei-ej) - f(x-ei+ej) + f(x-ei-ej))/(4*h[i]*h[j])
    return H

hstep = np.array([0.01, 0.01, 0.05, 0.01, 0.05])
H = hessian(lambda p: -lnpost(p), theta_map, hstep)
Sigma = np.linalg.inv(H); sig_map = np.sqrt(np.diag(Sigma))

In [ ]:
import emcee, arviz as az, corner

# ---- Settings, and the stopping rule written BEFORE looking at output ----
names  = ['M', 'alpha', 'beta', 'gamma', 'ln_sigma_int']
ndim, nw, nsteps, burn = 5, 32, 12000, 5000      # longer than before: Rhat was ~1.012
RHAT_MAX, ESS_MIN = 1.01, 400                     # run longer if Rhat > 1.01 or ESS < 400

theta_map, sig_map, lo, hi = (np.asarray(v, dtype=np.float64) for v in (theta_map, sig_map, lo, hi))

# ---- Four independent ensembles, each from its own overdispersed center ----
# Center = MAP + 8 * (curvature sigma) * N(0,1) per parameter, so every ensemble starts
# several posterior widths away in a different direction. Walkers form a small ball (0.5 sigma) around it.
chains = []
for k in range(4):
    while True:
        center = theta_map + 8 * sig_map * rng.standard_normal(ndim)
        if np.all((center > lo) & (center < hi)) and np.isfinite(lnpost(center)):
            break
    start = np.clip(center + 0.5 * sig_map * rng.standard_normal((nw, ndim)), lo + 1e-6, hi - 1e-6)
    s = emcee.EnsembleSampler(nw, ndim, lnpost)
    s.run_mcmc(start, nsteps, progress=False)
    chains.append(np.asarray(s.get_chain(), dtype=np.float64))      # (nsteps, nw, ndim)
    print(k, 'tau =', np.round(s.get_autocorr_time(tol=0), 0),
          'acceptance =', round(float(s.acceptance_fraction.mean()), 2))

arr  = np.concatenate([ch.transpose(1, 0, 2) for ch in chains])[:, burn:, :]   # (4*nw, draws, ndim)
flat = arr.reshape(-1, ndim)

# ---- Item 2: Rhat and ESS for every parameter ----
print('\nParameter      Rhat(128 walkers)  Rhat(4 indep, worst)  ESS_bulk  ESS_tail')
for i, n in enumerate(names):
    r_pool = float(az.rhat(arr[:, :, i]))
    r_worst = max(float(az.rhat(np.stack([ch[burn:, w, i] for ch in chains])))
                  for w in range(nw))
    eb = float(az.ess(arr[:, :, i])); et = float(az.ess(arr[:, :, i], method='tail'))
    flag = '  <-- run longer' if (r_pool > RHAT_MAX or eb < ESS_MIN) else ''
    print(f'{n:13s} {r_pool:10.4f} {r_worst:18.3f} {eb:12.0f} {et:9.0f}{flag}')
print('ensemble medians (should agree to << posterior width):')
for k, ch in enumerate(chains):
    print(' ', k, np.round(np.median(ch[burn:].reshape(-1, ndim), axis=0), 3))

# ---- Item 3: corner plot and percentile error bars ----
fig = corner.corner(flat, labels=['$M$ (mag)', r'$\alpha$', r'$\beta$', r'$\gamma$ (mag)', r'$\ln\sigma_{\rm int}$'],
                    quantiles=[.16, .5, .84], show_titles=True)
plt.show()
post = {n: flat[:, i] for i, n in enumerate(names)}
post['sigma_int'] = np.exp(flat[:, 4])
print()
for n in ['M', 'alpha', 'beta', 'gamma', 'sigma_int']:
    l, m, h = np.percentile(post[n], [16, 50, 84])
    print(f'{n:10s} = {m:.4f} +{h-m:.4f} -{m-l:.4f}')

# ---- Item 4: correlations, then fix M ----
corr = np.corrcoef(flat.T)
print('\ncorrelation matrix (M, alpha, beta, gamma, ln_sigma_int):\n', np.round(corr, 2))

sig_g_marg = flat[:, 3].std(ddof=1)
P = np.linalg.inv(np.cov(flat.T))                       # precision matrix of the samples
sig_g_fixM_samples = np.sqrt(np.linalg.inv(P[1:, 1:])[2, 2])   # drop M; gamma is index 2 afterwards
sig_g_fixM_hess    = np.sqrt(np.linalg.inv(H[1:, 1:])[2, 2])
print(f'sigma_gamma marginal        = {sig_g_marg:.4f} mag')
print(f'sigma_gamma | M fixed (samples) = {sig_g_fixM_samples:.4f} mag')
print(f'sigma_gamma | M fixed (Hessian) = {sig_g_fixM_hess:.4f} mag')
print(f'ratio fixed/marginal = {sig_g_fixM_samples/sig_g_marg:.2f}, '
      f'sqrt(1-rho_Mgamma^2) = {np.sqrt(1-corr[0,3]**2):.2f}')

**ANSWER:**

2. I would have run longer if R hat was above 1.01, or ESS was less tham 400, both of these for any perameter.
4. Gamma and M are strongly correlated, with a p of -0.77. M is brigthness, and M+ gamma is brightness in higher mass hosts. When M is more negative (star brighter), M must be offset by faint gamma to keep high mass fixed. There also may be some correlation between gamma and alpha, but that could just be because the data can't seperate them cleanly. 

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
lnlike0, lnpost0, lo0, hi0 = make_model(D, with_gamma=False)
x0_0 = np.delete(theta_map, 3)
fit0 = minimize(lambda p: -lnpost0(p) if np.isfinite(lnpost0(p)) else 1e10, x0_0,
                method='Nelder-Mead', options=dict(maxiter=20000, xatol=1e-6, fatol=1e-8))
H0 = hessian(lambda p: -lnpost0(p), fit0.x, np.delete(hstep, 3))

def laplace_lnZ(lnpost_, xhat, H_):
    k = len(xhat); _, logdet = np.linalg.slogdet(H_)
    return lnpost_(xhat) + 0.5*k*np.log(2*np.pi) - 0.5*logdet

lnZ1 = laplace_lnZ(lnpost,  theta_map, H)
lnZ0 = laplace_lnZ(lnpost0, fit0.x,    H0)
print('2 ln B10 =', 2*(lnZ1 - lnZ0))

# prior on gamma twice as wide
_, lnpost1w, _, _ = make_model(D, gmax=0.6)
print('2 ln B10 (2x prior) =', 2*(laplace_lnZ(lnpost1w, theta_map, H) - lnZ0))

# BIC
Nn = len(D['mB']); dlnL = lnlike(theta_map) - lnlike0(fit0.x)
print('BIC0 - BIC1 =', 2*dlnL - np.log(Nn))

**ANSWER:**

1. If the prior is twice as wide, so 0.6 (-0.3 to 0.3 mag) times 2, getting 1.2, than 2ln(b10) would equal -1.2 instead of 0.16. Shifts by -2ln2. Doubling flat prior halves denisty at nodes

3. it is weak, being only 0.16. The sign also changes when the prior is doubled.
Given gamma was 0.0524 +0.0239, with is 0.3 sigma away from the published value. A step of -0.06 is 2.5 sigma as well, so 250 can somewhat detect this step, its under 3 sigma soit is close. 

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

In [ ]:
%matplotlib inline
names = ['M', 'alpha', 'beta', 'gamma', 'ln_sigma_int']
def simulate(truth, rng, D):
    M, a, b, g, sint = truth
    n = len(D['z'])
    L = np.linalg.cholesky(D['Cov'])
    e = np.einsum('nij,nj->ni', L, rng.standard_normal((n, 3)))
    out = dict(D)
    out['x1'] = D['x1'] + e[:, 1]; out['c'] = D['c'] + e[:, 2]
    out['mB'] = (M + D['mu'] - a*D['x1'] + b*D['c'] + g*D['step'] + e[:, 0]
                 + rng.normal(0, np.hypot(sint, D['sig_pec'])))
    return out

truth = np.array([-19.3, 0.14, 3.0, -0.06, 0.10])        # your choice
tvec  = np.array([truth[0], truth[1], truth[2], truth[3], np.log(truth[4])])
nsim, rng = 100, np.random.default_rng(1)
est, err = [], []
for _ in range(nsim):
    S = simulate(truth, rng, D)
    lnl, lnp, lo_, hi_ = make_model(S)
    f = minimize(lambda p: -lnl(p), x0, method='L-BFGS-B', bounds=list(zip(lo_, hi_)))
    Hh = hessian(lambda p: -lnl(p), f.x, hstep)
    est.append(f.x); err.append(np.sqrt(np.diag(np.linalg.inv(Hh))))
est, err = np.array(est), np.array(err)
inside = np.abs(est - tvec) < err
cover = inside.mean(axis=0)
pull = (est - tvec)/err
for i, n in enumerate(names):
    print(f'{n}: coverage = {cover[i]:.2f}, pull mean = {pull[:,i].mean():.2f}, pull std = {pull[:,i].std():.2f}')

names = ['M', 'alpha', 'beta', 'gamma', 'ln_sigma_int']   # reset: a later cell overwrote this
%matplotlib inline

fig, axs = plt.subplots(1, 5, figsize=(16, 3.2))
zz = np.linspace(-4, 4, 200)
for i, ax in enumerate(axs):
    ax.hist(pull[:, i], bins=np.linspace(-4, 4, 17), density=True, alpha=0.6)
    ax.plot(zz, stats.norm.pdf(zz), 'k-', lw=1)
    ax.axvline(0, color='gray', ls=':', lw=0.8)
    ax.set_title(f'{names[i]}\nmean={pull[:,i].mean():.2f}, std={pull[:,i].std():.2f}', fontsize=10)
    ax.set_xlabel(r'pull $(\hat\theta-\theta_{\rm true})/\sigma$')
axs[0].set_ylabel('density')
plt.tight_layout(); plt.show()

**ANSWER:**

It would fail if the peaks in the histograms differ dramatically from thepredicted curves, and wqhen I ran the test, it appears beta and alpha clearly do not match. I then had claude rerun this again but removing some of the bias in thedata, potentially, especiallly for beta and alpha, and then it did mach much better to 68%

In [ ]:
def make_model_latent(dat, gmax=0.3):
    # theta = M, alpha, beta, gamma, ln_sint, xbar_lo, xbar_hi, cbar, ln_Rx, ln_Rc
    lo = np.array([-21, -1, -1, -gmax, np.log(1e-3), -3, -3, -0.5, np.log(0.1),  np.log(0.01)])
    hi = np.array([-17,  1,  8,  gmax, 0.0,           3,  3,  0.5, np.log(5.0),  np.log(1.0)])
    logvol = np.sum(np.log(hi - lo))
    y_obs = lambda g: np.stack([dat['mB'] - dat['mu'] - g*dat['step'], dat['x1'], dat['c']], axis=1)
    def lnlike(p):
        M, a, b, g, lns, xb0, xb1, cb, lnRx, lnRc = p
        xb = np.where(dat['step'] == 1, xb1, xb0)
        m  = np.stack([M - a*xb + b*cb, xb, np.full_like(xb, cb)], axis=1)
        A  = np.array([[-a, b], [1, 0], [0, 1]])
        R  = np.diag([np.exp(2*lnRx), np.exp(2*lnRc)])
        S  = dat['Cov'] + (A @ R @ A.T)[None]            # new array, safe to modify
        S[:, 0, 0] += dat['sig_pec']**2 + np.exp(2*lns)
        dy = y_obs(g) - m
        _, logdet = np.linalg.slogdet(S)
        sol = np.linalg.solve(S, dy[..., None])[..., 0]
        return -0.5*np.sum(np.einsum('ni,ni->n', dy, sol) + logdet + 3*np.log(2*np.pi))
    def lnpost(p):
        p = np.asarray(p)
        if np.any(p < lo) or np.any(p > hi): return -np.inf
        return lnlike(p) - logvol
    return lnlike, lnpost, lo, hi
def simulate_latent(truth, rng, D):
    M, a, b, g, sint, xb0, xb1, cb, Rx, Rc = truth
    n = len(D['z'])
    xb = np.where(D['step'] == 1, xb1, xb0)
    x1t = xb + Rx*rng.standard_normal(n)
    ct  = cb + Rc*rng.standard_normal(n)
    L = np.linalg.cholesky(D['Cov'])
    e = np.einsum('nij,nj->ni', L, rng.standard_normal((n, 3)))
    out = dict(D)
    out['x1'] = x1t + e[:, 1]
    out['c']  = ct  + e[:, 2]
    out['mB'] = (M + D['mu'] - a*x1t + b*ct + g*D['step'] + e[:, 0]
                 + rng.normal(0, np.hypot(sint, D['sig_pec'])))
    return out

# Population truth taken from the data: observed variance minus mean noise variance
Rx0 = np.sqrt(np.var(D['x1']) - np.mean(D['Cov'][:, 1, 1]))
Rc0 = np.sqrt(np.var(D['c'])  - np.mean(D['Cov'][:, 2, 2]))
truthL = np.array([-19.3, 0.14, 3.0, -0.06, 0.10,
                   D['x1'][D['step']==0].mean(), D['x1'][D['step']==1].mean(),
                   D['c'].mean(), Rx0, Rc0])
tvecL  = truthL.copy(); tvecL[4] = np.log(truthL[4]); tvecL[8] = np.log(Rx0); tvecL[9] = np.log(Rc0)
hstepL = np.array([0.01, 0.01, 0.05, 0.01, 0.05, 0.05, 0.05, 0.01, 0.05, 0.05])
x0L = tvecL + 0.0   # start somewhere other than the truth:
x0L[:5] = [-19.3, 0.10, 2.5, 0.0, np.log(0.1)]

namesL = ['M', 'alpha', 'beta', 'gamma', 'ln_sigma_int', 'xbar_lo', 'xbar_hi', 'cbar', 'ln_Rx', 'ln_Rc']
rng = np.random.default_rng(1)
estL, errL = [], []
for _ in range(100):
    S = simulate_latent(truthL, rng, D)
    lnl, lnp, lo_, hi_ = make_model_latent(S)
    f = minimize(lambda p: -lnl(p), x0L, method='L-BFGS-B', bounds=list(zip(lo_, hi_)))
    Hh = hessian(lambda p: -lnl(p), f.x, hstepL)
    estL.append(f.x); errL.append(np.sqrt(np.diag(np.linalg.inv(Hh))))
estL, errL = np.array(estL), np.array(errL)
pullL = (estL - tvecL)/errL
cover = np.mean(np.abs(estL - tvecL) < errL, axis=0)
for i, n in enumerate(namesL[:5]):
    print(f'{n}: coverage = {cover[i]:.2f}, pull mean = {pullL[:,i].mean():.2f}, pull std = {pullL[:,i].std():.2f}')
# Pull histograms for the latent-variable pipeline (same layout as the first test)
fig, axs = plt.subplots(1, 5, figsize=(16, 3.2))
zz = np.linspace(-4, 4, 200)
for i, ax in enumerate(axs):
    ax.hist(pullL[:, i], bins=np.linspace(-4, 4, 17), density=True, alpha=0.6)
    ax.plot(zz, stats.norm.pdf(zz), 'k-', lw=1)
    ax.axvline(0, color='gray', ls=':', lw=0.8)
    ax.set_title(f'{namesL[i]}\nmean={pullL[:,i].mean():.2f}, std={pullL[:,i].std():.2f}', fontsize=10)
    ax.set_xlabel(r'pull $(\hat\theta-\theta_{\rm true})/\sigma$')
axs[0].set_ylabel('density')
plt.tight_layout(); plt.show()

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

**REFEREE REPORT:**
1. The first error is that they used 3 sigma clipping, which throws out any data points 3 sigma away from the center. 
From the end plot, it is clear that the tails of the data stretch into the heavy mass stars end, and that this data isn't incorrect, rather that just is a trend in the data. Before even categorizing the stellar populations of two different groups, however, it already cut those data points out from the set. It removed the popularion the question was trying to answer in the first place. You can't cut out outliers and THEN ask if there were/are two groups.

2. It prints All R-hat < 1.1: the chains have converged no matter what. It codes exactly that, but they don't converge just by looking at it. The graphs go all over the place. When I ran the code to check this, it showed that the 1.1 number was not correct for any of the variables. Rhat was not less than 1.1 for multiple of the varibles. 

3.  The verification plan stars using the earlier found valuesit claims to be true, by calling "flat"
This code: best = np.median(flat, axis=0), assumes the starting points are the data it assumed were true (before verifying)
then, this code:
sim = np.concatenate([rng.normal(best[0], best[1], m.size - n2),
                      rng.normal(best[2], best[3], n2)]) Simulates the verifcation data based on the original values obtained, so its not providing a true verficatkion of the data, its generating a data set based on the "true" data it already found, and is thus baised towards the "flat" data it found. This test was actually negative, and the simuated data was almost the same as the true data found earlier. It would need to have multiple, randomly generated, clean data sets to actually work. I had that coded too, and when it was run, it confirmed it was still mostly correct this time, but the extended tests were still shakey. I would doadifferentverification plan entirely at this point or it signifies something is wrong in the modeland fix elsewhere. 


In [ ]:

# Simple demonstrations of errors 1 (clip), 3 (convergence), 4 (validation).
# Edit PATH, then run the whole thing (works in Jupyter or as a script).
import numpy as np, emcee
from scipy import optimize

PATH = "data/jhaff2/p3_wd_masses.csv"     # <-- edit this

d = np.genfromtxt(PATH, delimiter=",", names=True, dtype=None, encoding="utf-8")
M = d["mass_Msun"]
names = ["mu1", "sigma1", "mu2", "sigma2", "f"]
rng = np.random.default_rng(0)

def logp(p, m, ordered=False):
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    if ordered and not mu1 < mu2:
        return -np.inf
    a = np.log(f) - 0.5*((m-mu1)/s1)**2 - np.log(s1)
    b = np.log(1-f) - 0.5*((m-mu2)/s2)**2 - np.log(s2)
    return np.sum(np.logaddexp(a, b)) - 0.5*m.size*np.log(2*np.pi)

def ml_fit(m):                                  # best-fit by optimisation, several starts
    best = None
    for _ in range(15):
        p0 = [rng.uniform(.5,.65), rng.uniform(.02,.1), rng.uniform(.7,1.), rng.uniform(.04,.2), rng.uniform(.4,.9)]
        r = optimize.minimize(lambda p: -logp(p, m, True) if np.isfinite(logp(p, m, True)) else 1e12,
                              p0, method="Nelder-Mead", options=dict(maxiter=4000))
        if best is None or r.fun < best.fun: best = r
    return best.x

show = lambda p: "  ".join(f"{n}={v:.3f}" for n, v in zip(names, p))

# =====================================================================
print("ERROR 1: the 3-sigma clip")
print("=====================================================================")
med, sd = np.median(M), np.std(M)
keep = np.abs(M - med) < 3*sd
print(f"The clip keeps masses between {med-3*sd:.2f} and {med+3*sd:.2f}")
print(f"It removes {np.sum(~keep)} stars: {np.sum(M[~keep] > med)} heavy ones, {np.sum(M[~keep] < med)} light ones")
print("Masses of the removed stars:", np.round(np.sort(M[~keep]), 2))
print(f"Fraction of stars above 0.9 Msun: before clip {np.mean(M>0.9):.3f}, after clip {np.mean(M[keep]>0.9):.3f}")
print("Fit WITHOUT clip:", show(ml_fit(M)))
print("Fit WITH clip   :", show(ml_fit(M[keep])))

# =====================================================================
print("\nERROR 2: the convergence check")
print("=====================================================================")
m = M[keep]                                     # the notebook fits the clipped data
lo, hi = m[m < 0.7], m[m >= 0.7]
p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size/m.size])

def rhat(x):                                    # the notebook's formula
    n = x.shape[0]; B = n*np.var(x.mean(0), ddof=1); W = np.mean(np.var(x, 0, ddof=1))
    return np.sqrt(((n-1)/n*W + B/n)/W)

np.random.seed(42)
S = emcee.EnsembleSampler(32, 5, logp, args=(m,))
S.run_mcmc(p0 + 1e-3*np.random.randn(32, 5), 3000, progress=False)
ch = S.get_chain(discard=500)
print("R-hat from the notebook's own code (it printed 'converged' regardless):")
for i, n in enumerate(names): print(f"   {n:7s} {rhat(ch[:,:,i]):.3f}   (needs to be < 1.1)")
tau = S.get_autocorr_time(discard=500, tol=0)
print("Autocorrelation time (steps per independent sample):", np.round(tau, 0))
print(f"3000 steps is only {3000/tau.max():.0f} x the slowest tau; you want at least ~50")
print(f"Walkers that ended with mu1 > mu2 (labels swapped): {np.mean(ch[-1,:,0] > ch[-1,:,2]):.0%}")

# the fix: no clip, ordered means (mu1<mu2), walkers start spread out, many more steps
nw = 48
start = np.column_stack([rng.uniform(.45,.7,nw), rng.uniform(.02,.15,nw), rng.uniform(.7,1.1,nw),
                         rng.uniform(.03,.25,nw), rng.uniform(.2,.9,nw)])
S2 = emcee.EnsembleSampler(nw, 5, logp, args=(M,), kwargs=dict(ordered=True))
S2.run_mcmc(start, 15000, progress=False)
tau2 = S2.get_autocorr_time(tol=0)
print(f"\nFixed run: tau = {np.round(tau2,0)}, 15000 steps = {15000/tau2.max():.0f} x tau")
flat = S2.get_chain(discard=int(5*tau2.max()), flat=True)
print("Fixed-run answer (median):", show(np.median(flat, axis=0)))

# =====================================================================
print("\nERROR 3: the validation test")
print("=====================================================================")
best = np.median(flat, axis=0)
n2 = rng.binomial(M.size, 1 - best[4])
fake = np.concatenate([rng.normal(best[0], best[1], M.size-n2), rng.normal(best[2], best[3], n2)])

def refit(start_point, nsteps):
    s = emcee.EnsembleSampler(32, 5, logp, args=(fake,))
    s.run_mcmc(start_point + 1e-3*np.random.randn(32, 5), nsteps, progress=False)
    return np.median(s.get_chain(discard=nsteps//4, flat=True), axis=0)

lo2, hi2 = fake[fake < 0.7], fake[fake >= 0.7]
crude = np.array([lo2.mean(), lo2.std(), hi2.mean(), hi2.std(), lo2.size/fake.size])
print("TRUE values used to make the fake data:", show(best))
print("Notebook way (start AT the truth, 2000 steps):", show(refit(best, 2000)))
print("Same, but only 40 steps (no real fitting!)   :", show(refit(best, 40)))
print("Honest way (start from a crude guess, 2000)  :", show(refit(crude, 2000)))
print("If the 40-step run 'recovers' the truth too, the test cannot fail, so it proves nothing.")

In [ ]:
# Error 3, done properly: test the notebook's pipeline on MANY fake datasets.
# Edit PATH, then run. Takes a few minutes.
import numpy as np, emcee
from scipy import optimize

PATH = "data/jhaff2/p3_wd_masses.csv"     # <-- edit this
NSIM = 30                                      # fake datasets per test (raise for smoother numbers)

d = np.genfromtxt(PATH, delimiter=",", names=True, dtype=None, encoding="utf-8")
M = d["mass_Msun"]
names = ["mu1", "sigma1", "mu2", "sigma2", "f"]
rng = np.random.default_rng(0)
np.random.seed(0)

def logp(p, m):                                # same model and priors as the notebook
    mu1, s1, mu2, s2, f = p
    if not (0.3 < mu1 < 1.3 and 0.3 < mu2 < 1.3 and 0.005 < s1 < 0.5 and 0.005 < s2 < 0.5 and 0 < f < 1):
        return -np.inf
    a = np.log(f) - 0.5*((m-mu1)/s1)**2 - np.log(s1)
    b = np.log(1-f) - 0.5*((m-mu2)/s2)**2 - np.log(s2)
    return np.sum(np.logaddexp(a, b))

# --- the "truth" for the fake datasets: a good fit to your real data ---
best = None
for _ in range(15):
    p0 = [rng.uniform(.5,.65), rng.uniform(.02,.1), rng.uniform(.7,1.), rng.uniform(.04,.2), rng.uniform(.4,.9)]
    r = optimize.minimize(lambda p: -logp(p, M) if np.isfinite(logp(p, M)) else 1e12, p0, method="Nelder-Mead",
                          options=dict(maxiter=4000))
    if best is None or r.fun < best.fun: best = r
truth = best.x
if truth[0] > truth[2]:                        # put the lighter population first
    truth = np.array([truth[2], truth[3], truth[0], truth[1], 1 - truth[4]])
print("Truth used for the fake data:", dict(zip(names, np.round(truth, 3))))

def make_fake(contaminate):
    n2 = rng.binomial(M.size, 1 - truth[4])
    x = np.concatenate([rng.normal(truth[0], truth[1], M.size - n2), rng.normal(truth[2], truth[3], n2)])
    if contaminate:                            # a few odd stars, like real spectroscopic fits produce
        k = 10
        x[:k] = np.concatenate([rng.uniform(0.25, 0.45, k//2), rng.uniform(1.0, 1.3, k - k//2)])
    return x

def notebook_pipeline(x, clip):                # what the notebook does: (optional clip), crude start, 2000 steps
    if clip:
        x = x[np.abs(x - np.median(x)) < 3*np.std(x)]
    lo, hi = x[x < 0.7], x[x >= 0.7]
    p0 = np.array([lo.mean(), lo.std(), hi.mean(), hi.std(), lo.size/x.size])
    s = emcee.EnsembleSampler(32, 5, logp, args=(x,))
    s.run_mcmc(p0 + 1e-3*np.random.randn(32, 5), 2000, progress=False)
    c = s.get_chain(discard=500, flat=True)
    return np.median(c, axis=0), np.percentile(c, [16, 84], axis=0)

def run_test(label, contaminate, clip):
    est = np.zeros((NSIM, 5)); inside = np.zeros((NSIM, 5), bool)
    for k in range(NSIM):
        e, (lo, hi) = notebook_pipeline(make_fake(contaminate), clip)
        est[k] = e; inside[k] = (lo <= truth) & (truth <= hi)
    print(f"\n{label}   ({NSIM} fake datasets)")
    print(f"  {'':8s}{'truth':>8s}{'mean fit':>10s}{'bias':>8s}{'scatter':>9s}{'68% interval':>14s}")
    print(f"  {'':8s}{'':>8s}{'':>10s}{'':>8s}{'':>9s}{'holds truth':>14s}")
    for i, n in enumerate(names):
        print(f"  {n:8s}{truth[i]:8.3f}{est[:,i].mean():10.3f}{est[:,i].mean()-truth[i]:+8.3f}"
              f"{est[:,i].std():9.3f}{inside[:,i].mean():14.0%}")

print("\nA 68% interval should contain the truth about 68% of the time. Far below that = overconfident or biased.")
run_test("TEST 1: clean fake data, no clip (what the notebook's validation resembles)", False, False)
run_test("TEST 2: fake data with a few odd stars, notebook's 3-sigma clip applied (like your real data)", True, True)

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

1. I used claude to make all the code on this assignment. I also had it explain to me some of thee statistics, and I fed it all the info needed to format the equations properly. I had to manually correct the equations as it kept messing them up, forgetting a term or two, and it randomly squared something that shouldn't have been. Claude did help mewrite some of the write up answers,especially parts I didn't know how to word or explain correctly. When I didn't know something, I asked it to explain it to me in depth and confirmed with past labs and lecture slides if I needed to that it was right. 
2.  it had no idea what my data was, and it kept saying as much so, so it never ran the code itself. Ihad tolook at the outputs myself and decide if they matched what teh AI said or expected them to, if it said anything at all. It made some general stat statements that were wrong, using its predicted numbers, for example: in part 1 it was etimating what the numbers should be (without even seeing the data) and it was right for some, but others didn't match up at all. Since I caught this, and told it that, turns out the error bar it recomended me to trust was wrong. 

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.